# FPGA图像反转模块测试
## 第一阶段：数据通路验证

In [ ]:
from fpga_image_processor import ImageInverter
import numpy as np
import time

## 1. 初始化FPGA

In [ ]:
processor = ImageInverter('design_1.bit')

## 2. 基本功能测试

In [ ]:
# 测试全黑图像
black = np.zeros((100, 100), dtype=np.uint8)
result = processor.invert(black)
print(f"全黑图像测试: {'通过' if np.all(result == 255) else '失败'}")

# 测试全白图像
white = np.full((100, 100), 255, dtype=np.uint8)
result = processor.invert(white)
print(f"全白图像测试: {'通过' if np.all(result == 0) else '失败'}")

# 测试中灰图像
gray = np.full((100, 100), 128, dtype=np.uint8)
result = processor.invert(gray)
print(f"中灰图像测试: {'通过' if np.all(result == 127) else '失败'}")

## 3. 性能测试

In [ ]:
sizes = [(120, 160), (240, 320), (480, 640)]

for h, w in sizes:
    img = np.random.randint(0, 256, (h, w), dtype=np.uint8)
    
    times = []
    for _ in range(10):
        start = time.time()
        _ = processor.invert(img)
        times.append((time.time() - start) * 1000)
    
    avg_time = np.mean(times)
    fps = 1000 / avg_time
    
    print(f"{w}x{h}: {avg_time:.2f}ms, {fps:.2f}fps")

## 4. 验证正确性

In [ ]:
test_img = np.random.randint(0, 256, (480, 640), dtype=np.uint8)

# FPGA处理
result_fpga = processor.invert(test_img)

# CPU处理
result_cpu = 255 - test_img

# 对比
match = np.allclose(result_fpga, result_cpu)
print(f"FPGA与CPU结果一致: {match}")

if not match:
    diff = np.abs(result_fpga.astype(int) - result_cpu.astype(int))
    print(f"最大差异: {diff.max()}")

## 5. 性能统计

In [ ]:
stats = processor.get_performance_stats()
print(f"总处理次数: {stats['count']}")
print(f"平均处理时间: {stats['avg_time_ms']:.2f}ms")
print(f"平均帧率: {stats['fps']:.2f}fps")

## 测试完成
第一阶段验证通过，可以进入下一阶段（SAD模块开发）